# KION: воспроизводимый бейзлайн рекомендательной системы

Ноутбук реализует **корректный с методологической точки зрения бейзлайн** для исследования на датасете KION
(онлайн-кинотеатр, ~962 тыс. пользователей, ~15,7 тыс. объектов, ~5,5 млн взаимодействий, март–август 2021).

## Протокол оценки (без утечек данных)

* **Глобальный временной сплит** по дате просмотра: `train` (до 2021-08-08) / `valid` (7 дней) / `test` (7 дней).
  Случайное разбиение не используется: оно приводит к утечке будущего в признаки.
* **Все признаки и модели** для оценки на конкретном окне вычисляются **только по истории до начала этого окна**
  (принцип as-of): популярность, статистики пользователей/объектов, iALS-эмбеддинги, гендерные агрегаты.
* **Полный протокол оценки**, а не метрики на сэмплированных негативах: для каждого пользователя таргет-окна
  рекомендации выбираются из всего **warm-каталога** (объекты с хотя бы одним взаимодействием в истории) с фильтрацией
  уже просмотренного. Метрики: Precision@10, Recall@10, MAP@10, NDCG@10, MRR@10, Coverage@10.
* Внутри `train` выделяется **дополнительный OOF-срез** (последняя неделя) — на нём обучается ранкер,
  чтобы не использовать в обучении события из `valid`/`test`.

## Роль `random_seed` (воспроизводимость)

Временной протокол **не зависит от seed**: разбиение `train`/`valid`/`test` и все as-of признаки —
детерминированные сравнения меток времени (`datetime < valid_start` и т.п.), закреплённые
контрольными assert-ами (непересечение окон, полнота распределения строк). Seed управляет
только стохастическими компонентами:

* сэмплирование негативов в обучающих группах ранкера — `np.random.default_rng(seed)`;
* случайный бейзлайн `random` и FAST-подвыборка пользователей;
* инициализация iALS и стохастические попарные сравнения YetiRank / перестановки CatBoost.

Все перечисленные операции работают только с данными соответствующего исторического среза
(до as-of cutoff), поэтому seed не может занести информацию из будущего — его роль чисто
техническая: сделать стохастику воспроизводимой. Seed-ансамбль ранкеров (2×T4) отличается
**только** `random_seed`: обучающие данные, признаки и временной протокол у моделей идентичны,
усреднение скоров снижает дисперсию предсказаний; разброс метрик между моделями ансамбля
можно использовать как оценку чувствительности к seed.

## Сравниваемые модели

| # | Модель | Тип |
|---|--------|-----|
| 1 | Random | нижняя граница |
| 2 | Popularity (вся история) | неперсонализированный |
| 3 | Popularity-7d (свежая) | неперсонализированный |
| 4 | iALS (+popularity-fallback для холодных) | коллаборативная фильтрация |
| 5 | iALS+Pop кандидаты → CatBoost YetiRank | двухэтапная (итоговый бейзлайн) |

## Как запускать

1. Kaggle: *Create Notebook* → включить **Internet** в настройках (для скачивания датасета) и
   **GPU-ускоритель** (опционально, но полный прогон на CPU занимает часы). Если датасет уже добавлен как
   Kaggle Input — просто укажите `DATA_DIR` в ячейке загрузки на путь с `interactions.csv`.
2. Ресурсы полного прогона: **~10–12 ГБ RAM** (крупнейшие таблицы — признаки кандидатов ~17 млн строк);
   на Kaggle достаточно GPU-инстанса (T4×2 / P100) или CPU-инстанса (30 ГБ).
   На инстансе T4×2 обе карты используются параллельно: ранкер — seed-ансамбль
   из двух моделей, по одной на GPU (CatBoost не масштабирует одно обучение
   на несколько устройств), скоры на инференсе усредняются.
3. Первый прогон можно сделать с `CFG.fast_run = True` (подвыборка 50 тыс. пользователей, меньше итераций) —
   проверка пайплайна за несколько минут. Учтите: в FAST-режиме iALS сознательно ослаблен
   (3 итерации + разреженная подвыборка пользователей), его низкие метрики ожидаемы.
   Финальный прогон для статьи — `fast_run = False`.
4. Результаты и графики сохраняются в `./artifacts/`.

> Исправления относительно первой версии пайплайна: единая функция очистки (вместо трёх проходов),
> консистентное число негативов в обучающих/служебных группах ранкера (20), финальные метрики считаются по полному
> протоколу (а не на сэмплированных негативах), добавлены оценка на `test` с переобучением на `train+valid`,
> бейзлайны для сравнения и сохранение результатов. Мёртвый код (неиспользуемые ячейки признаков) удалён.

In [ ]:
# Установка зависимостей (в Kaggle catboost/tqdm уже есть, implicit — ставим)
%pip install -q implicit catboost tqdm

In [ ]:
import gc
import json
import os
import subprocess
import warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)


# ---------------------------------------------------------------------------
# Конфигурация эксперимента: все параметры в одном месте (воспроизводимость)
# ---------------------------------------------------------------------------
class CFG:
    seed = 20261020                  # единый seed для всего пайплайна
                                     # сплит от seed НЕ зависит: он детерминирован метками
                                     # времени; seed фиксирует только негативы, YetiRank,
                                     # random-бейзлайн и FAST-подвыборку (подробнее — во введении)

    # --- протокол оценки ---
    k_recommend = 10                 # длина итоговых рекомендаций
    n_train_negatives = 20            # негативов на пользователя в обучающих группах ранкера

    # --- кандидаты двухэтапной модели ---
    n_ials_candidates = 100           # кандидатов от iALS (тёплые пользователи)
    n_pop_candidates = 20             # добавка от популярности (тёплым)
    n_pop_candidates_cold = 100       # кандидаты от популярности (холодным)

    # --- iALS ---
    als_factors = 32
    als_iterations = 15
    als_regularization = 0.1

    # --- CatBoost ---
    # число моделей ранкера: по одной на GPU (2×T4), сиды идут подряд от seed
    cb_seed_ensemble = 2
    cb_iterations = 1000
    cb_early_stopping = 100
    cb_learning_rate = 0.05
    cb_depth = 8
    cb_l2_leaf_reg = 5

    # --- быстрый прогон для проверки пайплайна ---
    # ВНИМАНИЕ: в FAST-режиме iALS ослаблен (3 итерации + подвыборка пользователей),
    # его низкие метрики на подвыборке — ожидаемы и не характеризуют полный прогон
    fast_run = False                  # True: подвыборка пользователей + меньше итераций


if CFG.fast_run:
    CFG.als_iterations = 3
    CFG.cb_iterations = 30
    CFG.cb_early_stopping = 10
    CFG.user_sample = 50_000
else:
    CFG.user_sample = None

np.random.seed(CFG.seed)

ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

print(f"Конфигурация готова. FAST_RUN = {CFG.fast_run}, seed = {CFG.seed}")

# 1. Загрузка данных

Датасет KION скачивается из публичного репозитория (фиксированный commit — для воспроизводимости).
Схема: `interactions.csv` (user_id, item_id, last_watch_dt, total_dur, watched_pct), `users_en.csv`
(возраст, доход, пол, дети), `items_en.csv` (жанры, страны, год выпуска, возрастной рейтинг и др.).

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

from tqdm.auto import tqdm

COMMIT = "f69775be31fa5779907cf0a92ddedb70037fb5ae"
DATASET_URL = f"https://raw.githubusercontent.com/irsafilo/KION_DATASET/{COMMIT}/data_en.zip"

DATA_DIR = Path.cwd() / "data" / "data_en"
ARCHIVE_PATH = DATA_DIR.parent / "data_en.zip"


def download_progress(block_num, block_size, total_size):
    if total_size > 0:
        downloaded = min(block_num * block_size, total_size)
        progress_bar.total = total_size
        progress_bar.update(downloaded - progress_bar.n)


if not (DATA_DIR / "interactions.csv").exists():
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)

    if not ARCHIVE_PATH.exists():
        print("Скачивание data_en.zip…")
        with tqdm(
            total=0,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc="Загрузка",
        ) as progress_bar:
            urlretrieve(DATASET_URL, ARCHIVE_PATH, reporthook=download_progress)
        print("Скачивание завершено!")

    print("Распаковка архива…")
    with ZipFile(ARCHIVE_PATH) as archive:
        archive.extractall(DATA_DIR.parent)
    print("Распаковка завершена!")
else:
    print(f"Датасет уже загружен: {DATA_DIR}")

print(f"Каталог с данными: {DATA_DIR.resolve()}")

In [ ]:
DATA_DIR = Path.cwd() / "data" / "data_en"

INTERACTIONS_PATH = DATA_DIR / "interactions.csv"
USERS_PATH = DATA_DIR / "users_en.csv"
ITEMS_PATH = DATA_DIR / "items_en.csv"

for path in [INTERACTIONS_PATH, USERS_PATH, ITEMS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Не найден файл: {path}")


def load_metadata(path, id_name):
    """Загружает метаданные и восстанавливает ID из первого столбца, если нужно."""
    df = pd.read_csv(path, low_memory=False)
    unnamed = [c for c in df.columns if str(c).startswith("Unnamed:")]
    df = df.drop(columns=unnamed, errors="ignore")
    if id_name not in df.columns:
        df = df.rename(columns={df.columns[0]: id_name})
    return df


users = load_metadata(USERS_PATH, "user_id")
items = load_metadata(ITEMS_PATH, "item_id")
interactions = pd.read_csv(INTERACTIONS_PATH, low_memory=False)

if "last_watch_dt" in interactions.columns:
    interactions = interactions.rename(columns={"last_watch_dt": "datetime"})
interactions["datetime"] = pd.to_datetime(interactions["datetime"], errors="coerce")

# Режим быстрой проверки пайплайна: подвыборка пользователей (истории сохраняются целиком)
if CFG.user_sample is not None and interactions["user_id"].nunique() > CFG.user_sample:
    keep_users = (
        pd.Index(interactions["user_id"].unique())
        .to_series()
        .sample(CFG.user_sample, random_state=CFG.seed)
        .to_numpy()
    )
    interactions = interactions[interactions["user_id"].isin(keep_users)].reset_index(drop=True)
    users = users[users["user_id"].isin(keep_users)].reset_index(drop=True)
    items = items[items["item_id"].isin(interactions["item_id"].unique())].reset_index(drop=True)
    print(f"FAST_RUN: оставлено {CFG.user_sample:,} пользователей")

print("Загрузка завершена")
print(f"Interactions: {interactions.shape}")
print(f"Users:        {users.shape}")
print(f"Items:        {items.shape}")
print(f"Период: {interactions['datetime'].min().date()} — {interactions['datetime'].max().date()}")

In [ ]:
def audit_table(df, name):
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(3),
        "n_unique": df.nunique(dropna=True),
    })
    print(f"\n{'=' * 60}\n{name}: {df.shape[0]:,} строк × {df.shape[1]} столбцов")
    print(f"Полностью дублирующиеся строки: {df.duplicated().sum():,}")
    display(report.sort_values("missing_pct", ascending=False))


for df, name in [(interactions, "Interactions"), (users, "Users"), (items, "Items")]:
    audit_table(df, name)

print("\nУникальные ID:")
print("Users in interactions:", interactions["user_id"].nunique())
print("Items in interactions:", interactions["item_id"].nunique())
print("Users in metadata:", users["user_id"].nunique())
print("Items in metadata:", items["item_id"].nunique())

print("\nПроверка связей:")
print("User IDs без метаданных:", (~interactions["user_id"].isin(users["user_id"])).sum())
print("Item IDs без метаданных:", (~interactions["item_id"].isin(items["item_id"])).sum())

In [ ]:
# Ключевые распределения и проверки качества данных

# 1. Уникальность пар user-item (в KION пара встречается один раз — это важно для протокола)
pair_counts = interactions.groupby(["user_id", "item_id"], sort=False).size()
print("Уникальных пар user-item:", f"{len(pair_counts):,}")
print("Пар с несколькими строками:", f"{(pair_counts > 1).sum():,}")

# 2. Формат дат в сыром файле (ананомалии формата)
raw_date_col = "last_watch_dt" if "last_watch_dt" in pd.read_csv(INTERACTIONS_PATH, nrows=0).columns else "datetime"
raw_dates = pd.read_csv(INTERACTIONS_PATH, usecols=[raw_date_col], dtype="string")[raw_date_col].str.strip()
bad_format = raw_dates.notna() & ~raw_dates.str.fullmatch(r"\d{4}-\d{2}-\d{2}", na=False)
print("Дат с неожиданным форматом в сырых данных:", int(bad_format.sum()))

# 3. Статистики числовых полей
print("\nСтатистика total_dur:")
display(interactions["total_dur"].describe(percentiles=[0.01, 0.5, 0.9, 0.99]))
print("\nСтатистика watched_pct:")
display(interactions["watched_pct"].describe(percentiles=[0.01, 0.5, 0.9, 0.99]))

# 4. Активность во времени и распределения
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8), constrained_layout=True)

daily = interactions.groupby(interactions["datetime"].dt.date).size()
axes[0].plot(daily.index, daily.values, lw=1.2)
axes[0].set_title("Взаимодействия по дням")
axes[0].set_ylabel("событий/день")
axes[0].tick_params(axis="x", rotation=45)

user_activity = interactions.groupby("user_id").size()
axes[1].hist(np.log1p(user_activity), bins=60)
axes[1].set_title("Активность пользователей (log1p)")
axes[1].set_xlabel("log1p(объектов в истории)")

axes[2].hist(interactions["watched_pct"].dropna(), bins=50)
axes[2].set_title("Распределение watched_pct")
axes[2].set_xlabel("% просмотра")

plt.show()

# 2. Очистка данных (единый проход)

Очистка выполняется **одной функцией** `clean_kion_data` — это единственное место, где данные меняются
(в первой версии пайплайна очистка выполнялась трижды разными блоками — задваивала `_missing`-флаги
и усложняла воспроизведение).

Правила:
* удаление технических столбцов, полностью пустых столбцов, полных дубликатов строк;
* взаимодействия без `user_id` / `item_id` / даты удаляются;
* `total_dur <= 0` и `watched_pct` вне [0, 100] превращаются в пропуск (событие сохраняется);
* **экстремальные длительности** (больше всей длины периода наблюдений) удаляются;
* повторяющиеся `user_id`/`item_id` в метаданных → ошибка (не разбираем конфликты молча);
* `release_year` за пределами [1888, год наблюдений + 1] → пропуск;
* для каждого поля создаётся флаг `<col>_missing` — пропуски не «лечим» выдуманными значениями.

Важно: очистка не использует информацию из будущего — все правила опираются на период наблюдений целиком,
который известен и до разбиения (это свойство датасета, а не признак).

In [ ]:
import numpy as np
import pandas as pd


def clean_kion_data(interactions, users, items, drop_duration_over_period=True):
    """
    Очистка KION: interactions, users, items.

    Возвращает (interactions_clean, users_clean, items_clean).
    Пользователи из interactions не удаляются из-за отсутствия метаданных
    (этим управляет признак has_user_metadata на этапе моделирования).
    """

    # ---------- 1. Базовая очистка таблиц ----------

    def clean_table(df):
        df = df.copy()
        df.columns = df.columns.str.strip()

        unnamed = [c for c in df.columns if c.lower().startswith("unnamed:")]
        df = df.drop(columns=unnamed, errors="ignore")

        df = df.dropna(axis=1, how="all")  # полностью пустые столбцы

        null_tokens = {"", "na", "n/a", "nan", "none", "null", "<na>", "__missing__"}
        for col in df.select_dtypes(include=["object", "string"]).columns:
            s = df[col].astype("string").str.strip()
            mask = s.str.lower().isin(null_tokens)
            df[col] = s.mask(mask, pd.NA)

        n_duplicates = int(df.duplicated().sum())
        df = df.drop_duplicates().copy()
        return df, n_duplicates

    interactions, dup_interactions = clean_table(interactions)
    users, dup_users = clean_table(users)
    items, dup_items = clean_table(items)

    # ---------- 2. Проверка схемы ----------

    if "datetime" not in interactions.columns:
        if "last_watch_dt" in interactions.columns:
            interactions = interactions.rename(columns={"last_watch_dt": "datetime"})
        else:
            raise ValueError("В interactions отсутствует datetime/last_watch_dt")

    required = {"user_id", "item_id", "datetime"}
    if not required.issubset(interactions.columns):
        raise ValueError(f"Не хватает столбцов: {required - set(interactions.columns)}")

    if "user_id" not in users.columns or "item_id" not in items.columns:
        raise ValueError("В users/items отсутствует ключевой ID")

    # ---------- 3. Взаимодействия ----------

    interactions["datetime"] = pd.to_datetime(interactions["datetime"], errors="coerce")

    before_key_date_filter = len(interactions)
    interactions = interactions.dropna(subset=["user_id", "item_id", "datetime"]).copy()
    removed_key_date = before_key_date_filter - len(interactions)

    for col in ["total_dur", "watched_pct"]:
        if col not in interactions.columns:
            raise ValueError(f"Отсутствует обязательный столбец {col}")
        interactions[col] = pd.to_numeric(interactions[col], errors="coerce")

    # Невозможные значения -> пропуск (событие не удаляем)
    interactions.loc[interactions["total_dur"] <= 0, "total_dur"] = np.nan
    interactions.loc[~interactions["watched_pct"].between(0, 100), "watched_pct"] = np.nan

    interactions["total_dur_missing"] = interactions["total_dur"].isna().astype("int8")
    interactions["watched_pct_missing"] = interactions["watched_pct"].isna().astype("int8")

    # Экстремальные длительности: больше всего периода наблюдений -> удаляем
    removed_duration_outliers = 0
    if drop_duration_over_period and len(interactions):
        date_min = interactions["datetime"].min().normalize()
        date_max = interactions["datetime"].max().normalize()
        observation_days = (date_max - date_min).days + 1
        max_duration = observation_days * 24 * 60 * 60

        outlier_mask = interactions["total_dur"] > max_duration
        removed_duration_outliers = int(outlier_mask.sum())
        interactions = interactions.loc[~outlier_mask].copy()

    # В KION пара user-item уникальна; проверяем
    duplicate_pairs = int(interactions.duplicated(["user_id", "item_id"]).sum())
    if duplicate_pairs:
        raise ValueError(
            f"Найдено {duplicate_pairs:,} повторных пар user-item. "
            "Нужно проверить их до построения временных срезов."
        )

    # ---------- 4. Пользователи и объекты ----------

    users = users.dropna(subset=["user_id"]).copy()
    items = items.dropna(subset=["item_id"]).copy()

    for df, key, name in [(users, "user_id", "users"), (items, "item_id", "items")]:
        if df[key].duplicated(keep=False).any():
            raise ValueError(f"В {name} повторяются значения {key}")

    if "release_year" in items.columns:
        years = pd.to_numeric(items["release_year"], errors="coerce")
        latest_observed_year = interactions["datetime"].dt.year.max()
        valid_year = (
            years.notna()
            & years.between(1888, latest_observed_year + 1)
            & np.isclose(years.fillna(0) % 1, 0)
        )
        items["release_year"] = years.where(valid_year)

    # Индикаторы пропусков в метаданных
    for df, key in [(users, "user_id"), (items, "item_id")]:
        feature_cols = [c for c in df.columns if c != key and not c.endswith("_missing")]
        for col in feature_cols:
            df[f"{col}_missing"] = df[col].isna().astype("int8")

    # ---------- 5. Отчёт ----------

    print("ОЧИСТКА KION ЗАВЕРШЕНА")
    print("-" * 45)
    print(f"Точные дубликаты: interactions={dup_interactions:,}, users={dup_users:,}, items={dup_items:,}")
    print(f"Удалено взаимодействий без ключей/даты: {removed_key_date:,}")
    print(f"Удалено экстремальных длительностей: {removed_duration_outliers:,}")
    print(f"Повторных пар user-item: {duplicate_pairs:,}")
    print("\nИтоговые размеры:")
    print("interactions:", interactions.shape)
    print("users:", users.shape)
    print("items:", items.shape)
    print(
        "\nПользователей в interactions без профиля:",
        f"{len(set(interactions['user_id']) - set(users['user_id'])):,}",
    )

    return interactions, users, items

In [ ]:
interactions_clean, users_clean, items_clean = clean_kion_data(interactions, users, items)

del interactions, users, items
gc.collect()

# 3. Временной сплит

Разбиение строго по дате: последние 7 дней — `test`, предыдущие 7 дней — `valid`, всё до них — `train`.
Такой сплит имитирует продакшн-сценарий «обучаемся на прошлом, рекомендуем в будущее» и исключает утечку
будущих событий в признаки и модели.

In [ ]:
df = interactions_clean.copy()
df["datetime"] = pd.to_datetime(df["datetime"], errors="raise").dt.normalize()

# Последний день датасета
last_date = df["datetime"].max()

# Границы: 7 дней на test и 7 дней на validation
test_start = last_date - pd.Timedelta(days=6)
valid_start = test_start - pd.Timedelta(days=7)

train = df[df["datetime"] < valid_start].copy()
valid = df[(df["datetime"] >= valid_start) & (df["datetime"] < test_start)].copy()
test = df[df["datetime"] >= test_start].copy()

report = pd.DataFrame({
    "split": ["train", "valid", "test"],
    "start": [train["datetime"].min(), valid["datetime"].min(), test["datetime"].min()],
    "end": [train["datetime"].max(), valid["datetime"].max(), test["datetime"].max()],
    "rows": [len(train), len(valid), len(test)],
})
report["share_pct"] = (report["rows"] / len(df) * 100).round(2)
display(report)

# --- Гарантии протокола ---
assert len(train) + len(valid) + len(test) == len(df)
assert train["datetime"].max() < valid["datetime"].min()
assert valid["datetime"].max() < test["datetime"].min()


def assert_no_pair_overlap(a, b, name_a, name_b):
    """Пары user-item не должны пересекаться между окнами (иначе был бы дубль истории)."""
    merged = a[["user_id", "item_id"]].merge(b[["user_id", "item_id"]], on=["user_id", "item_id"], how="inner")
    assert merged.empty, f"Пары user-item пересекаются между {name_a} и {name_b}!"


assert_no_pair_overlap(train, valid, "train", "valid")
assert_no_pair_overlap(train, test, "train", "test")
assert_no_pair_overlap(valid, test, "valid", "test")

print("Разбиение успешно: все строки распределены без пересечений, пары user-item уникальны глобально.")

# interactions_clean больше не нужен: далее работаем только со срезами
del df, interactions_clean
gc.collect()

# 4. Протокол оценки

## Определения

* **Warm-каталог** для окна оценки = объекты с хотя бы одним взаимодействием в соответствующей истории
  (`train` для `valid`; `train + valid` для `test`). Только такие объекты имеют признаки и известны моделям.
* **Ground truth пользователя** = множество объектов из warm-каталога, с которыми он взаимодействовал
  в окне оценки. События с холодными объектами из оценки исключаются (модель не имеет возможности их
  порекомендовать) — их доля печатается для прозрачности.
* **Рекомендации** = топ-`K` объектов warm-каталога, **исключая уже просмотренные пользователем**
  (фильтрация seen — стандартная практика для implicit-рекомендаций).
* **Холодные пользователи** (без истории) не исключаются из оценки: неперсонализированные модели
  и двухэтапная модель обрабатывают их через популярность; для чистого iALS используется popularity-fallback
  (доля холодных печатается).

## Метрики (полный протокол)

Для каждого пользователя с непустым ground truth:

* **Precision@K** = |рекомендации ∩ релевантные| / K
* **Recall@K** = |рекомендации ∩ релевантные| / |релевантные|
* **AP@K** (average precision, знаменатель min(|релевантные|, K)) — MAP@K = среднее AP@K по пользователям
* **NDCG@K** — с бинарными релевантностями (дисконтирование 1/log2(rank+1), нормировка на идеальный DCG)
* **MRR@K** — обратный ранг первого релевантного
* **Coverage@K** — доля каталога, встречающаяся в рекомендациях (разнообразие моделей)

Все метрики усредняются по пользователям с непустым ground truth. **Метрики не считаются на сэмплированных
негативах** — негативы используются только внутри обучающих групп ранкера (YetiRank), что является стандартной
практикой обучения, но недопустимо для честной оценки качества.

In [ ]:
# ---------------------------------------------------------------------------
# Метрики полного протокола
# ---------------------------------------------------------------------------

def precision_at_k(recommended, relevant, k):
    hits = sum(1 for x in recommended[:k] if x in relevant)
    return hits / k


def recall_at_k(recommended, relevant, k):
    if not relevant:
        return 0.0
    hits = sum(1 for x in recommended[:k] if x in relevant)
    return hits / len(relevant)


def average_precision_at_k(recommended, relevant, k):
    if not relevant:
        return 0.0
    hits = 0
    precision_sum = 0.0
    for i, x in enumerate(recommended[:k]):
        if x in relevant:
            hits += 1
            precision_sum += hits / (i + 1)
    return precision_sum / min(len(relevant), k)


def ndcg_at_k(recommended, relevant, k):
    dcg = sum(1.0 / np.log2(i + 2) for i, x in enumerate(recommended[:k]) if x in relevant)
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0


def mrr_at_k(recommended, relevant, k):
    for i, x in enumerate(recommended[:k]):
        if x in relevant:
            return 1.0 / (i + 1)
    return 0.0


def evaluate_recommendations(recs, ground_truth, k, catalog_size=None):
    """Средние метрики по всем пользователям с непустым ground truth."""
    users = list(ground_truth)
    n_users = len(users)
    out = {"users": n_users}
    out[f"P@{k}"] = float(np.mean([precision_at_k(recs.get(u, []), ground_truth[u], k) for u in users]))
    out[f"R@{k}"] = float(np.mean([recall_at_k(recs.get(u, []), ground_truth[u], k) for u in users]))
    out[f"MAP@{k}"] = float(np.mean([average_precision_at_k(recs.get(u, []), ground_truth[u], k) for u in users]))
    out[f"NDCG@{k}"] = float(np.mean([ndcg_at_k(recs.get(u, []), ground_truth[u], k) for u in users]))
    out[f"MRR@{k}"] = float(np.mean([mrr_at_k(recs.get(u, []), ground_truth[u], k) for u in users]))
    out["avg_rec_len"] = float(np.mean([len(recs.get(u, [])) for u in users]))
    if catalog_size:
        recommended_items = {x for u in users for x in recs.get(u, [])}
        out[f"Coverage@{k}"] = len(recommended_items) / catalog_size
    return out


def candidate_recall(cands, ground_truth):
    """Полнота генератора кандидатов: доля релевантных объектов, попавших в кандидаты."""
    return float(np.mean([
        len(set(cands.get(u, [])) & rel) / len(rel) for u, rel in ground_truth.items()
    ]))


print("Метрики определены: P@K, R@K, MAP@K, NDCG@K, MRR@K, Coverage@K")

In [ ]:
def build_eval_pack(history, target):
    """
    Подготовка к оценке на target-окне по истории history.

    - warm-каталог: объекты с >= 1 взаимодействием в history;
    - ground truth: события target с warm-объектами, сгруппированные по пользователям;
    - seen: просмотренные в history объекты (фильтруются из рекомендаций).
    """
    warm_items = np.sort(history["item_id"].unique())
    warm_set = set(warm_items)

    seen = history.groupby("user_id")["item_id"].agg(set).to_dict()

    warm_target = target[target["item_id"].isin(warm_set)]
    cold_item_events = len(target) - len(warm_target)

    ground_truth = warm_target.groupby("user_id")["item_id"].agg(set).to_dict()
    eval_users = np.array(sorted(ground_truth))

    cold_users = int(sum(1 for u in eval_users if u not in seen))

    # Защита протокола: релевантные объекты не должны встречаться в истории пользователя
    overlap = sum(len(ground_truth[u] & seen.get(u, set())) for u in eval_users)
    assert overlap == 0, "Найдены таргет-объекты, уже просмотренные в истории!"

    print(f"Событий в окне: {len(target):,}")
    print(f"Отброшено событий с холодными объектами: {cold_item_events:,}")
    print(f"Пользователей с непустым ground truth: {len(eval_users):,}")
    print(f"Из них холодных (нет истории): {cold_users:,}")
    print(f"Warm-каталог: {len(warm_items):,} объектов")

    return {
        "warm_items": warm_items,
        "warm_set": warm_set,
        "seen": seen,
        "ground_truth": ground_truth,
        "eval_users": eval_users,
    }


print("Функция подготовки оценки готова")

# 5. Бейзлайны-ретриверы

Три семейства моделей, обучающихся **только на истории** (для valid — только `train`):

1. **Random** — равномерно случайные объекты warm-каталога (без просмотренных). Нижняя граница любых метрик.
2. **Popularity** — глобальный топ по числу взаимодействий (вся история) и **Popularity-7d** — топ по
   последним 7 дням истории (as-of = последний день истории). Ловит «свежесть» спроса.
3. **iALS** — implicit ALS на бинарной user-item матрице. Рекомендации — топ объектов по скалярному
   произведению факторов, **полный перебор каталога** (не сэмпл). Пользователям без фактора
   (холодным) выдаётся топ популярности — это фиксируется в названии модели: `iALS(+pop)`.

Для всех моделей фильтруются уже просмотренные объекты. Детерминизм обеспечивается фиксацией seed.

In [ ]:
class PopularityModel:
    """Топ объектов по числу взаимодействий в истории (опционально — в окне последних days дней)."""

    def __init__(self, days=None, name="popularity"):
        self.days = days
        self.name = name

    def fit(self, history):
        h = history
        if self.days is not None:
            as_of = h["datetime"].max()
            h = h[h["datetime"] > as_of - pd.Timedelta(days=self.days)]
        counts = h.groupby("item_id").size().rename("cnt").reset_index()
        # Сортировка детерминирована: сначала count, затем item_id
        counts = counts.sort_values(["cnt", "item_id"], ascending=[False, True])
        self.ranking_ = counts["item_id"].to_numpy()
        return self

    def recommend(self, users, seen, k):
        recs = {}
        for u in users:
            s = seen.get(u, set())
            out = []
            for it in self.ranking_:
                if it not in s:
                    out.append(int(it))
                    if len(out) >= k:
                        break
            recs[u] = out
        return recs


def recommend_random(users, seen, catalog, k, seed):
    """Равномерно случайные объекты каталога без просмотренных (детерминировано seed)."""
    rng = np.random.default_rng(seed)
    recs = {}
    for u in users:
        s = seen.get(u, set())
        picks = []
        attempts = 0
        while len(picks) < k and attempts < 50:
            draw = rng.choice(catalog, size=k, replace=False)
            for it in draw:
                if it not in s and it not in picks:
                    picks.append(int(it))
                    if len(picks) >= k:
                        break
            attempts += 1
        recs[u] = picks
    return recs


print("PopularityModel и recommend_random готовы")

In [ ]:
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares


def fit_ials(history, factors=CFG.als_factors, iterations=CFG.als_iterations,
             regularization=CFG.als_regularization, seed=CFG.seed):
    """
    Обучает implicit ALS на разреженной бинарной user-item матрице истории.
    ID отображаются в непрерывные индексы только для обучения модели.
    """
    users_idx = pd.Index(history["user_id"].drop_duplicates())
    items_idx = pd.Index(history["item_id"].drop_duplicates())

    user_to_idx = {user_id: idx for idx, user_id in enumerate(users_idx)}
    item_to_idx = {item_id: idx for idx, item_id in enumerate(items_idx)}

    rows = history["user_id"].map(user_to_idx).to_numpy()
    cols = history["item_id"].map(item_to_idx).to_numpy()
    values = np.ones(len(history), dtype=np.float32)

    user_item = csr_matrix(
        (values, (rows, cols)),
        shape=(len(users_idx), len(items_idx)),
        dtype=np.float32,
    )
    user_item.sum_duplicates()
    user_item.data[:] = 1.0  # бинарный сигнал независимо от возможных повторов

    model = AlternatingLeastSquares(
        factors=factors,
        iterations=iterations,
        regularization=regularization,
        random_state=seed,
        use_gpu=False,
    )
    model.fit(user_item, show_progress=True)

    return {
        "model": model,
        "user_to_idx": user_to_idx,
        "item_to_idx": item_to_idx,
        "user_item": user_item,
        "n_users": len(users_idx),
        "n_items": len(items_idx),
    }


def ials_top_lists(bundle, users, seen, k, popularity_order=None, chunk=2048):
    """
    Топ-k объектов по iALS-оценке (полный перебор каталога, чанками по пользователям).

    Просмотренные объекты исключаются. Холодным пользователям (нет фактора) — топ популярности
    (если передан popularity_order). Возвращает dict: user_id -> упорядоченный список item_id.
    """
    model = bundle["model"]
    user_to_idx = bundle["user_to_idx"]
    item_to_idx = bundle["item_to_idx"]
    user_item = bundle["user_item"]

    item_ids_arr = np.zeros(len(item_to_idx), dtype=np.int64)
    for item_id, idx in item_to_idx.items():
        item_ids_arr[idx] = item_id

    warm = [u for u in users if u in user_to_idx]
    cold = [u for u in users if u not in user_to_idx]
    recs = {}

    # Холодные пользователи: popularity-fallback
    if popularity_order is not None:
        for u in cold:
            s = seen.get(u, set())
            out = []
            for it in popularity_order:
                if it not in s:
                    out.append(int(it))
                    if len(out) >= k:
                        break
            recs[u] = out

    # Тёплые пользователи: полный скоринг каталога чанками
    item_factors = model.item_factors
    kk = min(k, len(item_to_idx))

    for start in range(0, len(warm), chunk):
        chunk_users = warm[start:start + chunk]
        rows_idx = np.array([user_to_idx[u] for u in chunk_users], dtype=np.int64)

        scores = model.user_factors[rows_idx] @ item_factors.T  # (chunk, n_items)

        # Маскируем просмотренное (позиции из обучающей матрицы)
        hist_block = user_item[rows_idx].tocoo()
        scores[hist_block.row, hist_block.col] = -np.inf

        # Топ-k по строкам
        part = np.argpartition(-scores, kth=kk - 1, axis=1)[:, :kk]
        part_scores = np.take_along_axis(scores, part, axis=1)
        order = np.argsort(-part_scores, axis=1, kind="stable")
        top = np.take_along_axis(part, order, axis=1)

        for r, u in enumerate(chunk_users):
            recs[u] = [int(item_ids_arr[j]) for j in top[r]]

    return recs


print("fit_ials и ials_top_lists готовы")

In [ ]:
# Пакет оценки на valid: история = train. Все модели ниже обучаются ТОЛЬКО на train.
valid_pack = build_eval_pack(train, valid)

pop_valid = PopularityModel(name="popularity").fit(train)
pop7_valid = PopularityModel(days=7, name="popularity_7d").fit(train)
ials_valid = fit_ials(train)

print("\nМодели для оценки на valid обучены (история: только train — утечек нет).")

In [ ]:
%%time

# Оценка бейзлайнов на valid (полный протокол)
RESULTS = []


def add_result(stage, model, recs, pack):
    row = {"stage": stage, "model": model}
    row.update(evaluate_recommendations(recs, pack["ground_truth"], CFG.k_recommend,
                                        catalog_size=len(pack["warm_items"])))
    RESULTS.append(row)
    return row


k = CFG.k_recommend

recs_random = recommend_random(valid_pack["eval_users"], valid_pack["seen"],
                               valid_pack["warm_items"], k, seed=CFG.seed)
add_result("valid", "random", recs_random, valid_pack)

recs_pop = pop_valid.recommend(valid_pack["eval_users"], valid_pack["seen"], k)
add_result("valid", "popularity", recs_pop, valid_pack)

recs_pop7 = pop7_valid.recommend(valid_pack["eval_users"], valid_pack["seen"], k)
add_result("valid", "popularity_7d", recs_pop7, valid_pack)

# iALS: сразу топ-100 — они же кандидаты для ранкера (бейзлайн = первые k)
ials_lists_valid = ials_top_lists(
    ials_valid, valid_pack["eval_users"], valid_pack["seen"],
    k=CFG.n_ials_candidates, popularity_order=pop_valid.ranking_,
)
recs_ials = {u: lst[:k] for u, lst in ials_lists_valid.items()}
add_result("valid", "iALS(+pop)", recs_ials, valid_pack)

display(pd.DataFrame(RESULTS).style.format(precision=4))

# 6. Двухэтапная модель: кандидаты → ранкер (CatBoost YetiRank)

## Архитектура

**Этап 1 — генерация кандидатов** (на инференсе): для каждого пользователя таргет-окна берём
`iALS top-100` (полный перебор каталога) ∪ `Popularity top-20` (глобальный топ без просмотренного).
Холодным пользователям — топ-100 популярности. Кандидаты гарантируют высокую полноту при разумной стоимости:
метрика **Candidate Recall@100** печатается и является «потолком» качества второго этапа.

**Этап 2 — реранкинг**: CatBoost-ранкер со списочной функцией потерь YetiRank перебирает кандидатов
по ~45 признакам (статистики пользователя/объекта из истории, популярность в окнах 1/7/30 дней,
гендерные агрегаты, iALS-оценка, метаданные). Итоговые рекомендации — топ-10 ранкера.
Ранкер — seed-ансамбль: модели с разными `random_seed` обучаются параллельно, по одной
на каждый GPU (CatBoost не масштабирует одно обучение на несколько устройств
в Python API), а на инференсе их скоры усредняются. Модели различаются **только** seed:
данные, признаки и временной протокол идентичны (см. «Роль `random_seed`» во введении).

> Сравнение с чистым iALS остаётся честным: кандидаты содержат iALS top-100, т.е. содержат и его top-10,
> поэтому ранкер может как повторить iALS, так и улучшить его.

## Протокол обучения (без утечек)

* **Обучающие группы** формируются на OOF-срезе внутри train: история = всё до `train - 7 дней`,
  таргет = последняя неделя train. Позитивы — все warm-объекты пользователя из таргет-недели,
  негативы — 20 случайных warm-объектов (без пересечения с историей и позитивами).
* **Служебная валидация для early stopping** — та же процедура на неделю позже (история = train,
  таргет = valid-неделя). Сэмплированные негативы здесь используются только как служебный сигнал
  для early stopping; **все финальные метрики считаются по полному протоколу** (раздел 4).
* Все признаки вычисляются по истории, завершающейся до таргет-недели (as-of), iALS обучается на той же истории.
* **Финальная модель для test** переобучается на большем объёме: история = весь train, таргет = valid-неделя
  (каждое событие train+valid используется либо как признак, либо как метка), число итераций фиксируется
  по best_iteration модели для valid. Оценка — на test, модели и признаки — только по train+valid.

In [ ]:
def make_rank_candidates(history, target, n_negatives=CFG.n_train_negatives, seed=CFG.seed):
    """
    Группы для обучения ранкера: таргет-неделя + случайные негативы.

    Каталог = warm-объекты history (объекты с >= 1 взаимодействием). Негативы не пересекаются
    ни с историей пользователя, ни с его позитивами.
    """
    history = history.copy()
    target = target.copy()
    history["datetime"] = pd.to_datetime(history["datetime"]).dt.normalize()
    target["datetime"] = pd.to_datetime(target["datetime"]).dt.normalize()

    catalog = np.sort(pd.Index(history["item_id"]).unique())
    catalog_set = set(catalog)

    seen_by_user = history.groupby("user_id")["item_id"].agg(set).to_dict()
    warm_target = target[target["item_id"].isin(catalog_set)]

    rng = np.random.default_rng(seed)
    records = []
    next_group = 0

    for user_id, group in warm_target.groupby("user_id", sort=False):
        positives = set(group["item_id"].unique())
        seen = seen_by_user.get(user_id, set())
        blocked = seen | positives

        negatives = set()
        attempts = 0
        while len(negatives) < n_negatives and attempts < 10:
            draws = rng.choice(catalog, size=max(128, 3 * n_negatives), replace=True)
            for item_id in draws:
                if item_id not in blocked:
                    negatives.add(item_id)
                    if len(negatives) >= n_negatives:
                        break
            attempts += 1

        # Редкий fallback, если не удалось набрать нужное число с первой попытки
        if len(negatives) < n_negatives:
            remaining = np.array([x for x in catalog if x not in blocked and x not in negatives])
            need = min(n_negatives - len(negatives), len(remaining))
            if need:
                negatives.update(rng.choice(remaining, size=need, replace=False).tolist())

        if not negatives:
            continue

        for item_id in positives:
            records.append((next_group, user_id, item_id, 1))
        for item_id in negatives:
            records.append((next_group, user_id, item_id, 0))
        next_group += 1

    result = pd.DataFrame(records, columns=["group_id", "user_id", "item_id", "label"])
    if result.empty:
        raise ValueError("Не получилось сформировать ranking groups.")
    return result


print("make_rank_candidates готов")

In [ ]:
USER_CATEGORICAL = ["age", "income", "sex", "kids_flg"]
ITEM_CATEGORICAL = ["content_type", "age_rating", "for_kids", "genres", "countries"]


def history_statistics(history, as_of, users):
    """Статистики пользователей/объектов/гендерные агрегаты — строго по history, as-of-дата."""

    h = history.copy()
    h["datetime"] = pd.to_datetime(h["datetime"]).dt.normalize()
    as_of = pd.Timestamp(as_of).normalize()

    h["total_dur"] = pd.to_numeric(h["total_dur"], errors="coerce")
    h["watched_pct"] = pd.to_numeric(h["watched_pct"], errors="coerce")
    h["_logdur"] = np.log1p(h["total_dur"].clip(lower=0))

    # Агрегаты пользователей
    user_stats = (
        h.groupby("user_id")
        .agg(
            user_history_count=("item_id", "size"),
            user_n_items=("item_id", "nunique"),
            user_mean_watched_pct=("watched_pct", "mean"),
            user_mean_logdur=("_logdur", "mean"),
            user_last_seen=("datetime", "max"),
        )
        .reset_index()
    )
    user_stats["user_recency_days"] = (as_of - user_stats["user_last_seen"]).dt.days.clip(lower=0)
    user_stats = user_stats.drop(columns="user_last_seen")

    # Агрегаты объектов
    item_stats = (
        h.groupby("item_id")
        .agg(
            popularity_all=("user_id", "size"),
            item_n_users=("user_id", "nunique"),
            item_mean_watched_pct=("watched_pct", "mean"),
            item_mean_logdur=("_logdur", "mean"),
            item_last_seen=("datetime", "max"),
        )
        .reset_index()
    )
    item_stats["item_recency_days"] = (as_of - item_stats["item_last_seen"]).dt.days.clip(lower=0)
    item_stats = item_stats.drop(columns="item_last_seen")

    # Популярность во временных окнах (заканчиваются на as_of — не заглядываем в будущее)
    for days in [1, 7, 30]:
        window = h[(h["datetime"] > as_of - pd.Timedelta(days=days)) & (h["datetime"] <= as_of)]
        counts = window.groupby("item_id").size().rename(f"popularity_{days}d")
        item_stats = item_stats.merge(counts, on="item_id", how="left", validate="one_to_one")
        item_stats[f"popularity_{days}d"] = item_stats[f"popularity_{days}d"].fillna(0)

    for col in ["popularity_all", "popularity_1d", "popularity_7d", "popularity_30d"]:
        if col not in item_stats:
            item_stats[col] = 0
        item_stats[f"log1p_{col}"] = np.log1p(item_stats[col])

    # Гендерные агрегаты (только по историческим событиям)
    sex_map = users[["user_id", "sex"]].drop_duplicates("user_id")

    hg = h[["user_id", "item_id", "watched_pct"]].merge(sex_map, on="user_id", how="left", validate="many_to_one")
    hg["sex_group"] = hg["sex"].astype("string").fillna("__MISSING__")
    hg["_high_watch"] = np.where(
        hg["watched_pct"].notna(), (hg["watched_pct"] > 90).astype(float), np.nan,
    )

    gender_item = (
        hg.groupby(["item_id", "sex_group"], observed=True)
        .agg(
            item_gender_count=("item_id", "size"),
            item_high_watch_rate_by_gender=("_high_watch", "mean"),
        )
        .reset_index()
    )

    item_totals = item_stats[["item_id", "popularity_all"]]
    gender_totals = hg.groupby("sex_group").size().rename("gender_total_count").reset_index()

    gender_item = (
        gender_item.merge(item_totals, on="item_id", how="left")
        .merge(gender_totals, on="sex_group", how="left")
    )
    gender_item["item_gender_share"] = (
        gender_item["item_gender_count"] / gender_item["popularity_all"].replace(0, np.nan)
    )
    gender_item["item_rate_within_gender"] = (
        gender_item["item_gender_count"] / gender_item["gender_total_count"].replace(0, np.nan)
    )

    return user_stats, item_stats, gender_item


def build_rank_features(candidates, history, as_of, users, items, ials_bundle):
    """
    Строит признаки для пар (user_id, item_id).

    Всё вычисляется по history до as_of. Метаданные users/items — справочники без временной привязки,
    их использование безопасно (это атрибуты объектов, а не будущие события).
    """
    user_stats, item_stats, gender_stats = history_statistics(history, as_of, users)

    data = candidates.copy()
    data = data.merge(user_stats, on="user_id", how="left", validate="many_to_one")
    data = data.merge(item_stats, on="item_id", how="left", validate="many_to_one")

    # Метаданные пользователей
    u_cols = ["user_id"] + [c for c in USER_CATEGORICAL if c in users.columns]
    u = users[u_cols].copy()
    u = u.rename(columns={c: f"u_{c}" for c in u.columns if c != "user_id"})

    data["has_user_metadata"] = data["user_id"].isin(users["user_id"]).astype("int8")
    data = data.merge(u, on="user_id", how="left", validate="many_to_one")

    for col in USER_CATEGORICAL:
        name = f"u_{col}"
        if name not in data:
            data[name] = "__MISSING__"
        data[f"{name}_missing"] = data[name].isna().astype("int8")
        data[name] = data[name].astype("string").fillna("__MISSING__")

    # Метаданные объектов (компактный набор)
    item_cols = [
        c for c in
        ["item_id", "content_type", "release_year", "age_rating", "for_kids", "genres", "countries"]
        if c in items.columns
    ]
    it = items[item_cols].copy()
    it = it.rename(columns={c: f"i_{c}" for c in item_cols if c != "item_id"})

    data = data.merge(it, on="item_id", how="left", validate="many_to_one")

    if "i_release_year" not in data:
        data["i_release_year"] = np.nan
    data["i_release_year"] = pd.to_numeric(data["i_release_year"], errors="coerce")
    data["i_release_year_missing"] = data["i_release_year"].isna().astype("int8")

    for col in ITEM_CATEGORICAL:
        name = f"i_{col}"
        if name not in data:
            data[name] = "__MISSING__"
        data[f"{name}_missing"] = data[name].isna().astype("int8")
        data[name] = data[name].astype("string").fillna("__MISSING__")

    # Гендерные агрегаты для пола кандидата-пользователя.
    # popularity_all и gender_total_count — служебные столбцы history_statistics:
    # они уже есть в data (из item_stats) — оставляем их только для расчёта долей внутри
    # history_statistics и не тащим в мердж (иначе получили бы popularity_all_x/_y
    # и молча потеряли бы признак).
    data["_sex_group"] = data["u_sex"].astype("string").fillna("__MISSING__")
    gender_merge = gender_stats.drop(
        columns=["popularity_all", "gender_total_count"], errors="ignore"
    )
    data = data.merge(
        gender_merge,
        left_on=["item_id", "_sex_group"],
        right_on=["item_id", "sex_group"],
        how="left",
        validate="many_to_one",
    )
    data = data.drop(columns=["_sex_group", "sex_group"], errors="ignore")

    # iALS-оценка (модель обучена на той же истории)
    u_idx = data["user_id"].map(ials_bundle["user_to_idx"])
    i_idx = data["item_id"].map(ials_bundle["item_to_idx"])

    data["ials_user_known"] = u_idx.notna().astype("int8")
    data["ials_item_known"] = i_idx.notna().astype("int8")

    score = np.full(len(data), np.nan, dtype=np.float32)
    known = u_idx.notna() & i_idx.notna()

    positions = np.flatnonzero(known.to_numpy())
    if len(positions):
        u_arr = u_idx.loc[known].astype("int64").to_numpy()
        i_arr = i_idx.loc[known].astype("int64").to_numpy()
        model = ials_bundle["model"]
        score[positions] = np.sum(
            model.user_factors[u_arr] * model.item_factors[i_arr],
            axis=1,
        )

    data["ials_available"] = np.isfinite(score).astype("int8")
    data["ials_score"] = np.nan_to_num(score, nan=0.0, posinf=0.0, neginf=0.0).astype("float32")

    # Счётчики: нет истории -> 0; средние остаются NaN (CatBoost обрабатывает их сам)
    count_cols = [
        c for c in data.columns
        if c.startswith("popularity_")
        or c.startswith("log1p_popularity_")
        or c in ["item_n_users", "user_history_count", "user_n_items"]
    ]
    data[count_cols] = data[count_cols].fillna(0)

    return data


print("history_statistics и build_rank_features готовы")

In [ ]:
def build_candidate_pairs(ials_lists, pop_ranking, users, seen):
    """
    Кандидаты ранкера: iALS top-N ∪ популярность (top-20 тёплым / top-100 холодным),
    без просмотренных объектов и дубликатов. Возвращает DataFrame [user_id, item_id].
    """
    records = []
    for u in users:
        s = seen.get(u, set())
        ials_part = ials_lists.get(u, [])[: CFG.n_ials_candidates]
        for it in ials_part:
            records.append((u, int(it)))

        cand_set = set(ials_part)
        budget = CFG.n_pop_candidates if ials_part else CFG.n_pop_candidates_cold
        added = 0
        for it in pop_ranking:
            if added >= budget:
                break
            if it not in s and it not in cand_set:
                records.append((u, int(it)))
                cand_set.add(it)
                added += 1

    return pd.DataFrame(records, columns=["user_id", "item_id"])


print("build_candidate_pairs готов")

In [ ]:
from catboost import CatBoostRanker, Pool

CATEGORICAL_FEATURES = (
    [f"u_{c}" for c in USER_CATEGORICAL]
    + [f"i_{c}" for c in ITEM_CATEGORICAL]
)

NUMERIC_FEATURES = [
    "has_user_metadata",
    "user_history_count", "user_n_items",
    "user_mean_watched_pct", "user_mean_logdur", "user_recency_days",

    "popularity_all", "popularity_1d", "popularity_7d", "popularity_30d",
    "log1p_popularity_all", "log1p_popularity_1d",
    "log1p_popularity_7d", "log1p_popularity_30d",
    "item_n_users", "item_mean_watched_pct", "item_mean_logdur",
    "item_recency_days",

    "item_gender_share", "item_rate_within_gender",
    "item_high_watch_rate_by_gender", "item_gender_count",

    "ials_score", "ials_available", "ials_user_known", "ials_item_known",
    "i_release_year", "i_release_year_missing",
]

NUMERIC_FEATURES += [f"u_{col}_missing" for col in USER_CATEGORICAL]
NUMERIC_FEATURES += [f"i_{col}_missing" for col in ITEM_CATEGORICAL]

FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES


def prepare_rank_frame(frame):
    """Приводит признаки к типам CatBoost; проверяет наличие всех столбцов (fail-fast)."""
    missing = [c for c in FEATURES if c not in frame.columns]
    if missing:
        raise ValueError(f"Отсутствуют признаки (проверьте сборку): {missing}")

    # NaN в категориях уже заполнены в build_rank_features; category заметно экономит память
    for col in CATEGORICAL_FEATURES:
        frame[col] = frame[col].fillna("__MISSING__").astype("category")

    for col in NUMERIC_FEATURES:
        frame[col] = pd.to_numeric(frame[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

    return frame


def make_pool(frame):
    """Pool для CatBoostRanker: сортировка по group_id обязательна."""
    frame = frame.sort_values("group_id").reset_index(drop=True)
    return Pool(
        frame[FEATURES],
        label=frame["label"],
        group_id=frame["group_id"],
        cat_features=CATEGORICAL_FEATURES,
    )


def _list_gpu_devices():
    """Список индексов GPU (nvidia-smi); пустой список = GPU недоступен."""
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=15,
        )
        if out.returncode != 0:
            return []
        return [ln.strip() for ln in out.stdout.splitlines() if ln.strip()]
    except Exception:
        return []


CB_DEVICES = _list_gpu_devices()            # напр. ["0", "1"] на Kaggle T4 x2
CB_TASK_TYPE = "GPU" if CB_DEVICES else "CPU"

# Ансамбль ранкеров. Важно: CatBoost обучает ОДНУ модель ровно на ОДНОМ
# устройстве («devices="0:1"» НЕ ускоряет одно обучение в Python API), поэтому
# две T4 загружаем так: по модели на карту с разными random_seed
# (YetiRank стохастичен, модели получаются разными), а на инференсе
# скоры усредняются — seed-ensemble. На CPU ансамбль обучается так же.
CB_SEEDS = [CFG.seed + i for i in range(max(1, CFG.cb_seed_ensemble))]

if CB_TASK_TYPE == "GPU":
    CB_DEVICE_FOR = {s: CB_DEVICES[i % len(CB_DEVICES)] for i, s in enumerate(CB_SEEDS)}
    N_PARALLEL = min(len(CB_SEEDS), len(CB_DEVICES))
else:
    CB_DEVICE_FOR = {}
    N_PARALLEL = min(len(CB_SEEDS), max(1, (os.cpu_count() or 2) // 4))

# ядра CPU делим между параллельными обучениями (у GPU-обучения есть CPU-часть)
CB_THREADS_PER_MODEL = max(1, (os.cpu_count() or 2) // N_PARALLEL)

print(f"CatBoost task_type: {CB_TASK_TYPE}, devices: {CB_DEVICES or '-'}")
print(f"Ансамбль ранкеров: {len(CB_SEEDS)} шт. (seeds={CB_SEEDS}), параллельных обучений: {N_PARALLEL}")
print(f"Признаков всего: {len(FEATURES)} (категориальных: {len(CATEGORICAL_FEATURES)})")

In [ ]:
%%time

# ---------------------------------------------------------------------------
# Обучающие данные ранкера (валидационная стадия)
# ---------------------------------------------------------------------------

# OOF-срез внутри train: история до 2021-08-01, таргет — последняя неделя train
train_end = train["datetime"].max()
train_feature_cutoff = train_end - pd.Timedelta(days=7)

train_history = train[train["datetime"] <= train_feature_cutoff].copy()
train_target = train[(train["datetime"] > train_feature_cutoff) & (train["datetime"] <= train_end)].copy()

print(f"train_history: {train_history.shape}, train_target: {train_target.shape}")

# Группы для обучения
train_candidates = make_rank_candidates(train_history, train_target, seed=CFG.seed)

# iALS для признаков обучающих групп — обучен только на train_history
ials_for_train_features = fit_ials(train_history)

train_rank = build_rank_features(
    train_candidates,
    history=train_history,
    as_of=train_feature_cutoff,
    users=users_clean,
    items=items_clean,
    ials_bundle=ials_for_train_features,
)
train_rank = prepare_rank_frame(train_rank)

# Служебная валидация для early stopping: история = train, таргет = valid-неделя
valid_candidates = make_rank_candidates(train, valid, seed=CFG.seed + 1)

valid_rank = build_rank_features(
    valid_candidates,
    history=train,
    as_of=train_end,               # as-of = конец train (до начала valid)
    users=users_clean,
    items=items_clean,
    ials_bundle=ials_valid,        # iALS обучен на train — см. раздел 5
)
valid_rank = prepare_rank_frame(valid_rank)

display(pd.DataFrame([
    {"split": name, "rows": len(frame), "groups": frame["group_id"].nunique(),
     "positives": int(frame["label"].sum()), "negatives": int((frame["label"] == 0).sum())}
    for name, frame in [("train(OOF)", train_rank), ("valid(early-stop)", valid_rank)]
]))

train_pool = make_pool(train_rank)
valid_pool = make_pool(valid_rank)

# Пулы держат собственные копии данных — исходные фреймы больше не нужны
del train_candidates, valid_candidates, ials_for_train_features
del train_rank, valid_rank
gc.collect()

In [ ]:
%%time

# ---------------------------------------------------------------------------
# Обучение ансамбля YetiRank-ранкеров: по одной модели на каждый GPU (2×T4)
#
# CatBoost в Python обучает ОДНУ модель ровно на ОДНОМ устройстве: передать
# «devices="0:1"» и ускорить одно обучение в два раза нельзя (мульти-GPU для
# одной модели существует только в CLI/MPI-режиме). Поэтому две T4 заняты
# так: параллельно обучаются модели с разными random_seed (YetiRank
# стохастичен, модели получаются разными), а на инференсе их скоры
# усредняются (seed-ensemble). Время — как у одной модели, обе карты
# загружены, качество обычно не ниже одиночного ранкера.
#
# Замечание для статьи: GPU-обучение CatBoost недетерминировано (порядок
# CUDA-редукций) — при фиксированных seed допустим разброс в младших знаках
# метрик между прогонами.
# ---------------------------------------------------------------------------


def fit_ranker(seed, gpu_id):
    """Обучает один YetiRank-ранкер на указанном GPU (или на CPU)."""
    m = CatBoostRanker(
        loss_function="YetiRank",
        eval_metric="NDCG:top=10",

        task_type=CB_TASK_TYPE,
        **({"devices": str(gpu_id)} if CB_TASK_TYPE == "GPU" else {}),

        iterations=CFG.cb_iterations,
        depth=CFG.cb_depth,
        learning_rate=CFG.cb_learning_rate,
        l2_leaf_reg=CFG.cb_l2_leaf_reg,

        random_seed=seed,                      # у каждой модели ансамбля свой seed

        metric_period=50,
        verbose=50,                             # логи обучений чередуются — это нормально
        allow_writing_files=False,
        thread_count=CB_THREADS_PER_MODEL,     # ядра CPU делятся между обучениями
    )
    m.fit(
        train_pool,
        eval_set=valid_pool,
        early_stopping_rounds=CFG.cb_early_stopping,
        use_best_model=True,
    )
    return m


# CatBoost отпускает GIL на время fit(), поэтому потоки обучаются действительно
# параллельно на разных GPU и не дублируют данные (Pool читается обоими).
with ThreadPoolExecutor(max_workers=N_PARALLEL) as ex:
    rankers = list(ex.map(lambda s: fit_ranker(s, CB_DEVICE_FOR.get(s)), CB_SEEDS))

for seed, m in zip(CB_SEEDS, rankers):
    print(f"--- seed={seed} | best_iteration={m.best_iteration_}")
    best_score = m.get_best_score()
    if best_score:
        print("    служебный NDCG на группах с негативами:", best_score)
    else:
        print("    eval-метрики не сохранены (не критично: финальные метрики — полным протоколом)")

# Обучающий пул больше не нужен (valid_pool понадобится для финального переобучения)
del train_pool
gc.collect()


In [ ]:
%%time

# ---------------------------------------------------------------------------
# Двухэтапная модель на valid: кандидаты -> признаки -> ранкер -> топ-10
# ---------------------------------------------------------------------------

cand_pairs_valid = build_candidate_pairs(
    ials_lists_valid, pop_valid.ranking_, valid_pack["eval_users"], valid_pack["seen"],
)
print("Кандидатов:", f"{len(cand_pairs_valid):,}")

cands_valid_dict = cand_pairs_valid.groupby("user_id")["item_id"].apply(list).to_dict()
print(f"Candidate Recall@K (потолок ранкера):",
      f"{candidate_recall(cands_valid_dict, valid_pack['ground_truth']):.4f}")

# Признаки кандидатов: история = train, as-of = конец train, iALS из раздела 5
cand_features_valid = build_rank_features(
    cand_pairs_valid,
    history=train,
    as_of=train["datetime"].max(),
    users=users_clean,
    items=items_clean,
    ials_bundle=ials_valid,
)
cand_features_valid = prepare_rank_frame(cand_features_valid)


def predict_topk(models, frame, k=CFG.k_recommend, chunk=2_000_000):
    """Скорит пары ранкером или ансамблем ранкеров (средний скор) — top-k на пользователя."""
    if not isinstance(models, (list, tuple)):
        models = [models]

    scores = np.empty(len(frame), dtype=np.float32)
    data = frame[FEATURES]
    for start in range(0, len(frame), chunk):
        stop = min(start + chunk, len(frame))
        part = data.iloc[start:stop]
        # средний скор ансамбля (predict ранкера возвращает сырые значения формулы)
        scores[start:stop] = np.mean([m.predict(part) for m in models], axis=0)

    top = (
        pd.DataFrame({
            "user_id": frame["user_id"].to_numpy(),
            "item_id": frame["item_id"].to_numpy(),
            "score": scores,
        })
        .sort_values(["user_id", "score"], ascending=[True, False], kind="mergesort")
        .groupby("user_id", sort=False)
        .head(k)
    )
    return top.groupby("user_id")["item_id"].apply(list).to_dict()


recs_twostage_valid = predict_topk(rankers, cand_features_valid)
add_result("valid", "iALS+Pop -> CatBoost", recs_twostage_valid, valid_pack)

del cand_pairs_valid, cand_features_valid
gc.collect()

display(pd.DataFrame(RESULTS).style.format(precision=4))


# 7. Финальная оценка на test (переобучение на train + valid)

Протокол финального измерения:

* все модели и признаки пересчитываются по истории **train + valid** (iALS переобучается, популярность
  пересчитывается, статистики — as-of от 2021-08-15);
* **финальный ранкер** обучается на большем объёме: история = весь `train`, таргет = `valid`-неделя
  (это те же группы, что служили early-stop выборкой — теперь они становятся обучающими, что является
  стандартной практикой переобучения после выбора конфигурации). Число итераций фиксируется по
  `best_iteration` каждой модели ансамбля, отобранным на valid;
* оценка — на `test`-окне, которое не использовалось нигде выше.

Числа на test — те, что идут в статью; числа на valid — для анализа и model selection.

In [ ]:
%%time

# ---------------------------------------------------------------------------
# История финальной стадии: train + valid
# ---------------------------------------------------------------------------
test_history = pd.concat([train, valid], ignore_index=True)
test_as_of = test_history["datetime"].max()

test_pack = build_eval_pack(test_history, test)

# Модели, обученные на train + valid
pop_test = PopularityModel(name="popularity").fit(test_history)
pop7_test = PopularityModel(days=7, name="popularity_7d").fit(test_history)
ials_test = fit_ials(test_history)

k = CFG.k_recommend

recs_random_test = recommend_random(test_pack["eval_users"], test_pack["seen"],
                                     test_pack["warm_items"], k, seed=CFG.seed + 100)
add_result("test", "random", recs_random_test, test_pack)

recs_pop_test = pop_test.recommend(test_pack["eval_users"], test_pack["seen"], k)
add_result("test", "popularity", recs_pop_test, test_pack)

recs_pop7_test = pop7_test.recommend(test_pack["eval_users"], test_pack["seen"], k)
add_result("test", "popularity_7d", recs_pop7_test, test_pack)

ials_lists_test = ials_top_lists(
    ials_test, test_pack["eval_users"], test_pack["seen"],
    k=CFG.n_ials_candidates, popularity_order=pop_test.ranking_,
)
recs_ials_test = {u: lst[:k] for u, lst in ials_lists_test.items()}
add_result("test", "iALS(+pop)", recs_ials_test, test_pack)

# ---------------------------------------------------------------------------
# Финальный ансамбль ранкеров: те же сиды и раскладка по GPU, что и на valid;
# число итераций каждой модели зафиксировано по её best_iteration на valid
# ---------------------------------------------------------------------------
best_iters = [
    (m.best_iteration_ + 1) if m.best_iteration_ is not None else CFG.cb_iterations
    for m in rankers
]
best_iters = [max(b, 50) for b in best_iters]


def fit_final_ranker(seed, gpu_id, n_iters):
    m = CatBoostRanker(
        loss_function="YetiRank",
        task_type=CB_TASK_TYPE,
        **({"devices": str(gpu_id)} if CB_TASK_TYPE == "GPU" else {}),
        iterations=n_iters,
        depth=CFG.cb_depth,
        learning_rate=CFG.cb_learning_rate,
        l2_leaf_reg=CFG.cb_l2_leaf_reg,
        random_seed=seed,
        metric_period=50,
        verbose=50,
        allow_writing_files=False,
        thread_count=CB_THREADS_PER_MODEL,
    )
    m.fit(valid_pool)   # эти группы: признаки из train, метки — valid-неделя
    return m


with ThreadPoolExecutor(max_workers=N_PARALLEL) as ex:
    final_rankers = list(
        ex.map(lambda a: fit_final_ranker(*a),
               [(s, CB_DEVICE_FOR.get(s), b) for s, b in zip(CB_SEEDS, best_iters)])
    )

print("Финальные ранкеры обучены, итераций (выбрано на valid):",
      dict(zip(CB_SEEDS, best_iters)))

# ---------------------------------------------------------------------------
# Кандидаты и признаки для test
# ---------------------------------------------------------------------------
cand_pairs_test = build_candidate_pairs(
    ials_lists_test, pop_test.ranking_, test_pack["eval_users"], test_pack["seen"],
)
print("Кандидатов:", f"{len(cand_pairs_test):,}")

cands_test_dict = cand_pairs_test.groupby("user_id")["item_id"].apply(list).to_dict()
print("Candidate Recall@K (потолок ранкера):",
      f"{candidate_recall(cands_test_dict, test_pack['ground_truth']):.4f}")

cand_features_test = build_rank_features(
    cand_pairs_test,
    history=test_history,
    as_of=test_as_of,
    users=users_clean,
    items=items_clean,
    ials_bundle=ials_test,
)
cand_features_test = prepare_rank_frame(cand_features_test)

recs_twostage_test = predict_topk(final_rankers, cand_features_test)
add_result("test", "iALS+Pop -> CatBoost", recs_twostage_test, test_pack)

# Освобождаем крупнейшие промежуточные объекты (финальная ячейка результатов их не использует)
del rankers, cand_pairs_test, cand_features_test, ials_lists_test, ials_lists_valid, test_history
gc.collect()


# 8. Сводные результаты

Итоговая таблица — основной артефакт для статьи: все модели на обоих окнах оценки, единый полный
протокол. Также сохраняются: CSV с результатами, важности признаков финального ранкера, график
сравнения моделей и сводка конфигурации.

In [ ]:
results_df = pd.DataFrame(RESULTS)

metric_cols = [c for c in results_df.columns if c.startswith(("P@", "R@", "MAP@", "NDCG@", "MRR@"))]

print("=== VALID (model selection) ===")
display(results_df[results_df["stage"] == "valid"].set_index("model")[metric_cols].style.format(precision=4))

print("=== TEST (финальные числа для статьи) ===")
display(results_df[results_df["stage"] == "test"].set_index("model")[metric_cols].style.format(precision=4))

display(results_df.set_index(["stage", "model"])[["users", "avg_rec_len", f"Coverage@{CFG.k_recommend}"]])

# --- График сравнения ---
plot_metrics = [f"NDCG@{CFG.k_recommend}", f"MAP@{CFG.k_recommend}"]
model_order = (results_df[results_df["stage"] == "test"]
               .sort_values(plot_metrics[0], ascending=False)["model"].tolist())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), constrained_layout=True, sharey=True)
for ax, metric in zip(axes, plot_metrics):
    tmp = results_df.pivot(index="model", columns="stage", values=metric).reindex(model_order)
    tmp.plot(kind="bar", ax=ax, width=0.75, legend=metric == plot_metrics[0])
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
fig.suptitle("Сравнение моделей: valid и test (полный протокол)", fontsize=13)
plt.show()

# --- Важность признаков: среднее по финальному ансамблю ---
importance_values = np.mean(
    [m.get_feature_importance(valid_pool) for m in final_rankers], axis=0
)
importance = (
    pd.DataFrame({"feature": FEATURES, "importance": importance_values})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
print("Топ-20 признаков финального ранкера (среднее по ансамблю):")
display(importance.head(20))

In [ ]:
# --- Сохранение артефактов ---
results_df.to_csv(ARTIFACTS_DIR / "baseline_results.csv", index=False)
importance.to_csv(ARTIFACTS_DIR / "feature_importances.csv", index=False)

summary = {
    "dataset": "KION (data_en, commit f69775be)",
    "protocol": {
        "split": "temporal: train < valid_start, valid = 7d, test = last 7d",
        "as_of_features": "history only (no future information)",
        "ground_truth": "warm items of eval window, seen filtered",
        "metrics": "P@10, R@10, MAP@10, NDCG@10, MRR@10, Coverage@10 (full protocol)",
    },
    "config": {
        "seed": CFG.seed,
        "k_recommend": CFG.k_recommend,
        "n_train_negatives": CFG.n_train_negatives,
        "n_ials_candidates": CFG.n_ials_candidates,
        "n_pop_candidates": CFG.n_pop_candidates,
        "als": {"factors": CFG.als_factors, "iterations": CFG.als_iterations,
                "regularization": CFG.als_regularization},
        "catboost": {"loss": "YetiRank",
                     "iterations_chosen_on_valid": [int(b) for b in best_iters],
                     "seed_ensemble": len(final_rankers), "ensemble_seeds": list(CB_SEEDS),
                     "depth": CFG.cb_depth, "lr": CFG.cb_learning_rate,
                     "l2_leaf_reg": CFG.cb_l2_leaf_reg, "task_type": CB_TASK_TYPE,
                     "n_gpus": len(CB_DEVICES)},
        "fast_run": CFG.fast_run,
        "user_sample": CFG.user_sample,
    },
    "results": results_df.to_dict(orient="records"),
}
with open(ARTIFACTS_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=str)

print(f"Артефакты сохранены в {ARTIFACTS_DIR.resolve()}:")
for p in sorted(ARTIFACTS_DIR.iterdir()):
    print(" -", p.name, f"({p.stat().st_size:,} байт)")

# 9. Ограничения бейзлайна и направления развития (для обсуждения в статье)

**Ограничения протокола** (осознанные, зафиксированные):

1. **Холодные объекты** — объекты без взаимодействий в истории исключены из ground truth и не могут
   быть рекомендованы (warm-start постановка). Доля отброшенных событий печатается в `build_eval_pack`.
2. **Один период обучения ранкера** — ранкер учится на одной таргет-неделе (rolling-режим с несколькими
   периодами обычно улучшает устойчивость, но усложняет бейзлайн).
3. **Распределение негативов**: при обучении негативы сэмплируются равномерно из warm-каталога, а на
   инференсе ранкер переранжирует кандидатов iALS/популярности (более «сложные» примеры). Это стандартная
   практика для первого бейзлайна, но вариант с негативами из кандидатов стоит проверить.
4. **Валидация и early stopping** выполнялись на тех же данных, что и обучение финального ранкера
   (valid-неделя) — это не влияет на test-оценку, но числа valid слегка оптимистичны.
5. **Гиперпараметры не тюнировались** (кроме отбора числа итераций через early stopping) — важно
   зафиксировать в статье, чтобы приросты от улучшений нельзя было объяснить настройкой.

**Направления развития**: мультинедельное обучение ранкера; учёт watched_pct/total_dur в лейблах и
весах (weighted ALS); BM25/TF-IDF ретривер и ANN-поиск по эмбеддингам; другие лоссы ранкера
(QuerySoftMax, LambdaMART); тюнинг iALS и CatBoost; холодный старт контента по метаданным;
онлайн-оценка A/B.

**Что уже даёт этот бейзлайн**: корректный временной протокол без утечек, воспроизводимость
(фиксированные seed и commit датасета), честные полно-каталожные метрики и таблица сравнения
пяти моделей — достаточная опора для формулирования исследовательских гипотез.